# Mel unet high frequency discriminator training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook tests a high-frequency discriminator for mel-domain refinement.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip -q install soundfile librosa

In [ ]:
import os, json, time, random
from pathlib import Path
from dataclasses import dataclass
from types import SimpleNamespace
from itertools import cycle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from IPython.display import Audio, display

print("torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

 ## Configuration

In [ ]:
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/master")
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"

# Choose mode:
#  - "A1"   = reconstruction only
#  - "A2"   = single-scale mel PatchGAN
#  - "A2MS" = multi-scale mel PatchGAN
MODE = "A2MS"

# Enforce 70/30 sampling
P_SPEECH = 0.70

# Inpainting k choices
K_CHOICES = (1, 2)

# Training
STEPS       = 40000
LOG_EVERY   = 100
CKPT_EVERY  = 500
SEG_S       = 3.0
BATCH       = 4
NUM_WORKERS = 2
PIN_MEMORY  = (device.type == "cuda")

# Generator architecture
G_BASE        = 24
G_GROUPS      = 8
G_USE_MASK    = True

# Optimizer / resume
LR_G = 2e-4
LR_D = 1e-4
RESUME_LR_SCALE = 0.5
AUTO_RESUME_SAME_FAMILY = False

# Core mel-domain reconstruction losses
LAMBDA_RECON      = 20.0
LAMBDA_HF         = 50.0
LAMBDA_TIME_DIFF  = 20.0
LAMBDA_DELTA      = 0.02
USE_IDENTITY_LOSS_FOR_K0 = True
LAMBDA_ID         = 0.10

# High-frequency weighting profile
HF_START_FRAC = 0.45
HF_END_GAIN   = 8.0
HF_RAMP_POWER = 2.0

# Boundary-aware loss masks
HF_MASK_DILATE    = 2
TDIFF_MASK_DILATE = 2

# Main mel GAN
LAMBDA_FM        = 5.0
LAMBDA_ADV_MAX   = 0.6
ADV_WARMUP_STEPS = 10000
D_UPDATE_EVERY   = 2

# New targeted high-frequency discriminator
USE_HF_DISCRIMINATOR = True
HF_D_START_FRAC      = 0.55   # only the upper 45% of mel bins
HF_D_BASE            = 24
HF_D_SCALES          = (1, 2, 4)
HF_D_ADV_MULT        = 0.75
HF_D_FM_MULT         = 1.00
HF_D_DLOSS_MULT      = 1.00

# BigVGAN frontend only (no frozen-vocoder aux loss in this fork)
BIGVGAN_MODEL_ID        = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False
USE_VOCODER_AUX         = False

# Run folder
RUN_TAG  = time.strftime("%Y%m%d_%H%M%S")
RUN_NAME = "mel_unet_high_frequency_discriminator"
CKPT_DIR = DRIVE_ROOT / "checkpoints_runs" / f"{RUN_NAME}_{RUN_TAG}"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV = CKPT_DIR / "train_log.csv"

print("MODE:", MODE)
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("LOG_CSV:", LOG_CSV)


In [ ]:
RUN_CONFIG = dict(
    mode=MODE,
    seed=SEED,
    split_dir=str(SPLIT_DIR),
    prepared_dataset_dir=str(PREPARED_DATASET_DIR),
    p_speech=P_SPEECH,
    k_choices=list(K_CHOICES),
    steps=STEPS,
    log_every=LOG_EVERY,
    ckpt_every=CKPT_EVERY,
    seg_s=SEG_S,
    batch=BATCH,
    num_workers=NUM_WORKERS,
    # generator
    G_arch="MelRefinerUNet2D",
    G_base=G_BASE,
    G_groups=G_GROUPS,
    G_use_mask=G_USE_MASK,
    # optimizer / resume
    lr_g=LR_G,
    lr_d=LR_D if MODE in ["A2", "A2MS"] else None,
    resume_lr_scale=RESUME_LR_SCALE,
    auto_resume_same_family=AUTO_RESUME_SAME_FAMILY,
    # mel-domain losses
    lambda_recon=LAMBDA_RECON,
    lambda_hf=LAMBDA_HF,
    lambda_time_diff=LAMBDA_TIME_DIFF,
    lambda_delta=LAMBDA_DELTA,
    use_identity_loss_for_k0=USE_IDENTITY_LOSS_FOR_K0,
    lambda_id=LAMBDA_ID,
    hf_start_frac=HF_START_FRAC,
    hf_end_gain=HF_END_GAIN,
    hf_ramp_power=HF_RAMP_POWER,
    hf_mask_dilate=HF_MASK_DILATE,
    tdiff_mask_dilate=TDIFF_MASK_DILATE,
    # GAN
    lambda_fm=LAMBDA_FM,
    lambda_adv_max=LAMBDA_ADV_MAX,
    adv_warmup_steps=ADV_WARMUP_STEPS,
    d_update_every=D_UPDATE_EVERY,
    use_hf_discriminator=USE_HF_DISCRIMINATOR,
    hf_d_start_frac=HF_D_START_FRAC,
    hf_d_base=HF_D_BASE,
    hf_d_scales=list(HF_D_SCALES),
    hf_d_adv_mult=HF_D_ADV_MULT,
    hf_d_fm_mult=HF_D_FM_MULT,
    hf_d_dloss_mult=HF_D_DLOSS_MULT,
    # frontend
    bigvgan_model_id=BIGVGAN_MODEL_ID,
    bigvgan_use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
    use_vocoder_aux=USE_VOCODER_AUX,
    vocoder_backend="BigVGAN",
    # optimizer defaults
    opt_g=dict(lr=LR_G, betas=(0.9, 0.99), weight_decay=1e-4),
    opt_d=dict(lr=LR_D, betas=(0.8, 0.99), weight_decay=1e-4) if MODE in ["A2", "A2MS"] else None,
)

(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("Saved run_config.json")


## BigVGAN mel frontend 
This variant uses the BigVGAN mel frontend so the spectrogram parameters stay aligned with the existing listening endpoint.

To keep this notebook light enough for training, BigVGAN itself is loaded lazily only when quick listening is enabled near the end.


In [ ]:
# ----------------------------
# BigVGAN repo + official mel frontend
# ----------------------------
BIGVGAN_DIR = Path("/content/BigVGAN")

if not BIGVGAN_DIR.exists():
    import subprocess, sys
    subprocess.run(
        ["git", "clone", "https://github.com/NVIDIA/BigVGAN", "/content/BigVGAN"],
        check=True,
    )

req_file = BIGVGAN_DIR / "requirements.txt"
if req_file.exists():
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)],
        check=True,
    )

import sys
if str(BIGVGAN_DIR) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_DIR))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

# Parameters for: nvidia/bigvgan_v2_22khz_80band_fmax8k_256x
# Kept explicit here so training does not need to keep the full vocoder on GPU.
h = SimpleNamespace(
    sampling_rate=22050,
    num_mels=80,
    n_fft=1024,
    hop_size=256,
    win_size=1024,
    fmin=0,
    fmax=8000,
)

SR = int(h.sampling_rate)
N_MELS = int(h.num_mels)

print("BigVGAN frontend ready.")
print("SR:", SR, "N_MELS:", N_MELS)
print("mel params:", dict(
    n_fft=h.n_fft,
    hop_size=h.hop_size,
    win_size=h.win_size,
    fmin=h.fmin,
    fmax=h.fmax,
))

def mel_spectrogram(
    wav: torch.Tensor,
    n_fft=None,
    num_mels=None,
    sampling_rate=None,
    hop_size=None,
    win_size=None,
    fmin=None,
    fmax=None,
    center: bool = False,
):
    return bigvgan_mel_spectrogram(
        wav,
        n_fft=h.n_fft if n_fft is None else n_fft,
        num_mels=h.num_mels if num_mels is None else num_mels,
        sampling_rate=h.sampling_rate if sampling_rate is None else sampling_rate,
        hop_size=h.hop_size if hop_size is None else hop_size,
        win_size=h.win_size if win_size is None else win_size,
        fmin=h.fmin if fmin is None else fmin,
        fmax=h.fmax if fmax is None else fmax,
        center=center,
    )

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is None:
        print("Loading BigVGAN for listening:", BIGVGAN_MODEL_ID)
        bigvgan_G = bigvgan.BigVGAN._from_pretrained(
            model_id=BIGVGAN_MODEL_ID,
            revision="main",
            cache_dir=None,
            force_download=False,
            proxies=None,
            resume_download=False,
            local_files_only=False,
            token=None,
            map_location=str(device),
            strict=False,
            use_cuda_kernel=False,
        )
        if hasattr(bigvgan_G, "remove_weight_norm"):
            try:
                bigvgan_G.remove_weight_norm()
            except Exception as e:
                print("remove_weight_norm warning:", e)
        bigvgan_G = bigvgan_G.eval().to(device)
        for p in bigvgan_G.parameters():
            p.requires_grad_(False)
    return bigvgan_G


In [ ]:
# Sanity check
with torch.no_grad():
    y_hat = bigvgan_G(torch.zeros(1, N_MELS, 10, device=device))
print("BigVGAN dummy output shape:", tuple(y_hat.shape))

In [ ]:
# =========================
# Stage audio to local disk for faster training.
# =========================

import shutil
from pathlib import Path

DRIVE_SPLIT_DIR = SPLIT_DIR
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

def _read_lines(p: Path):
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [ln for ln in lines if ln]

def to_local_path(p: Path) -> Path:
    """
    Preserve folder structure if possible.
    Tries to make the local path relative to DRIVE_ROOT or /content/drive.
    Falls back to a flat folder if neither works.
    """
    p = Path(p)
    for root in [DRIVE_ROOT, Path("/content/drive")]:
        try:
            rel = p.relative_to(root)
            return LOCAL_DATA_ROOT / rel
        except Exception:
            pass
    # fallback (may collide if duplicate filenames exist)
    return LOCAL_DATA_ROOT / "flat" / p.name

# Collect all referenced files
all_files = []
for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    assert src_list.exists(), f"Missing split list: {src_list}"
    all_files += [Path(x) for x in _read_lines(src_list)]

all_files = sorted(set(all_files))
print("Unique audio files referenced by splits:", len(all_files))

# Copy to /content (fast disk)
copied = 0
skipped = 0
for src in all_files:
    dst = to_local_path(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        skipped += 1
        continue
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} files (skipped {skipped} already present).")
print("Local cache root:", LOCAL_DATA_ROOT)

# Rewrite split lists to local paths
for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    src_paths = [Path(x) for x in _read_lines(src_list)]
    dst_paths = [str(to_local_path(p)) for p in src_paths]
    (LOCAL_SPLIT_DIR / name).write_text("\n".join(dst_paths) + "\n", encoding="utf-8")

print("Wrote local split lists to:", LOCAL_SPLIT_DIR)
SPLIT_DIR = LOCAL_SPLIT_DIR
print("Now using SPLIT_DIR:", SPLIT_DIR)

RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
RUN_CONFIG["local_data_root"] = str(LOCAL_DATA_ROOT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("Updated run_config.json with local split_dir")

# Verify local split lists exist and contain /content/ paths
for name in ["speech_train.txt","music_train.txt","speech_val.txt","music_val.txt"]:
    p = SPLIT_DIR / name
    assert p.exists(), f"Missing local list: {p}"
    first = p.read_text().splitlines()[0].strip().strip('"').strip("'")
    print(name, "first:", first)
    assert first.startswith("/content/"), "List still points to Drive paths!"
    assert Path(first).exists(), "First local wav missing!"
print(" Using local wav paths.")

In [ ]:
# =========================
# Dataset + loaders (speech/music separate, infinite iterators)
# =========================
from torch.utils.data import Dataset, DataLoader

def read_list(p: Path):
    assert p.exists(), f"Missing list file: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr: int, segment_s: float):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(sr * segment_s)
        assert len(self.paths) > 0, f"Empty list: {list_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        w = safe_load_wav_mono(self.paths[idx], self.sr)
        if w.numel() < self.seg_len:
            w = F.pad(w, (0, self.seg_len - w.numel()))
        max_start = w.numel() - self.seg_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        return w[start:start+self.seg_len]

# Split list paths after optional local staging
speech_train = SPLIT_DIR / "speech_train.txt"
music_train  = SPLIT_DIR / "music_train.txt"
speech_val   = SPLIT_DIR / "speech_val.txt"
music_val    = SPLIT_DIR / "music_val.txt"

ds_speech = FileListDataset(speech_train, SR, SEG_S)
ds_music  = FileListDataset(music_train,  SR, SEG_S)


dl_speech = DataLoader(ds_speech, batch_size=BATCH, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
                       persistent_workers=(NUM_WORKERS > 0))
dl_music  = DataLoader(ds_music,  batch_size=BATCH, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
                       persistent_workers=(NUM_WORKERS > 0))

def infinite(dl):
    while True:
        for b in dl:
            yield b

it_speech = infinite(dl_speech)
it_music  = infinite(dl_music)

def next_mixed_batch():
    b = next(it_speech) if random.random() < P_SPEECH else next(it_music)
    return b.to(device)

print("speech files:", len(ds_speech), "music files:", len(ds_music), "P_SPEECH:", P_SPEECH)

# we make sure batch is coming from /content paths
print("Example speech path:", read_list(speech_train)[0])

## Inpainting corruption + frequency-weighted / boundary-aware losses 
This stays close to the existing mel inpainting setup. The main new ingredient is an audio-side auxiliary loss computed after frozen BigVGAN.


In [ ]:
def linear_time_inpaint(mel: torch.Tensor, k: int, offset: int):
    """
    mel: (B, n_mels, T)
    k: number of missing frames between anchors
       - k=1 => anchor, missing, anchor, missing, ...
       - k=2 => anchor, missing, missing, anchor, ...
       - k=0 => no missing frames
    offset: shift pattern in [0, k]
    """
    B, C, T = mel.shape
    step = k + 1

    mel_interp = mel.clone()
    mask = torch.zeros((B, 1, T), device=mel.device, dtype=mel.dtype)

    if k == 0:
        return mel_interp, mask

    for left in range(offset, T - step, step):
        right = left + step
        for j in range(1, step):
            t = left + j
            alpha = j / step
            mel_interp[:, :, t] = (1.0 - alpha) * mel[:, :, left] + alpha * mel[:, :, right]
            mask[:, :, t] = 1.0

    return mel_interp, mask


def make_inpainting_pair(mel_real: torch.Tensor, k_choices=(1, 2), randomize_offset=True):
    k = int(random.choice(list(k_choices)))
    step = k + 1
    offset = int(random.randrange(step)) if randomize_offset else 0
    mel_interp, mask = linear_time_inpaint(mel_real, k=k, offset=offset)
    return {"real": mel_real, "interp": mel_interp, "mask": mask, "k": k, "offset": offset}


def build_freq_weights(
    n_mels: int,
    start_frac: float = HF_START_FRAC,
    end_gain: float = HF_END_GAIN,
    power: float = HF_RAMP_POWER,
    device=None,
):
    """
    Returns weights shaped (1, n_mels, 1), with low bins near 1.0 and
    a smooth ramp up toward 'end_gain' in the upper mel bins.
    """
    idx = torch.linspace(0.0, 1.0, steps=n_mels, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    weights = 1.0 + (end_gain - 1.0) * ramp
    return weights.view(1, n_mels, 1)


def dilate_mask_time(mask: torch.Tensor, radius: int):
    """
    mask: (B,1,T)
    expands the masked region in time by 'radius' frames on each side.
    """
    if radius <= 0:
        return mask
    k = 2 * radius + 1
    x = mask.float()
    x = F.max_pool1d(x, kernel_size=k, stride=1, padding=radius)
    return x.clamp_(0.0, 1.0)


HF_FREQ_WEIGHTS = build_freq_weights(N_MELS, device=device)


def masked_weighted_l1(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    freq_weights: torch.Tensor | None = None,
    w_missing: float = 1.0,
    w_anchor: float = 0.0,
    mask_dilate: int = 0,
):
    """
    Weighted L1 over (B, F, T). By default only inside missing frames,
    but can be expanded to include boundary frames via mask_dilate.
    """
    if mask_dilate > 0:
        mask = dilate_mask_time(mask, radius=mask_dilate)

    m = mask.expand(-1, pred.shape[1], -1)
    w = w_anchor + (w_missing - w_anchor) * m
    if freq_weights is not None:
        w = w * freq_weights
    err = (pred - target).abs()
    return (w * err).sum() / (w.sum() + 1e-8)


def masked_weighted_diff_l1(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    dim: int = -1,
    freq_weights: torch.Tensor | None = None,
    mask_dilate: int = 0,
):
    """
    L1 on first differences, restricted to regions touching the hole.

    dim = -1  -> time differences
    dim =  1  -> frequency differences
    """
    if mask_dilate > 0:
        mask = dilate_mask_time(mask, radius=mask_dilate)

    if dim == -1:
        dp = pred[..., 1:] - pred[..., :-1]
        dt = target[..., 1:] - target[..., :-1]
        dm = torch.maximum(mask[..., 1:], mask[..., :-1]).expand_as(dp)
        fw = freq_weights
    elif dim == 1:
        dp = pred[:, 1:, :] - pred[:, :-1, :]
        dt = target[:, 1:, :] - target[:, :-1, :]
        dm = mask.expand(-1, pred.shape[1], -1)
        dm = torch.maximum(dm[:, 1:, :], dm[:, :-1, :])
        fw = freq_weights[:, 1:, :] if freq_weights is not None else None
    else:
        raise ValueError("dim must be -1 (time) or 1 (frequency)")

    w = dm
    if fw is not None:
        w = w * fw
    err = (dp - dt).abs()
    return (w * err).sum() / (w.sum() + 1e-8)


def improvement_ratio(base: float, ref: float):
    return ref / (base + 1e-8) if base > 0 else float("nan")


## Generator G - residual 2D U-Net with axial-attention bottleneck 
This is a controlled architecture upgrade over the existing 2D U-Net:
- same mel-domain residual inpainting idea
- same mask conditioning
- adds axial self-attention only in the bottleneck, so the model receives longer-range time/frequency context without becoming a full transformer

In [ ]:
# Previous plain 1D residual refiner left out on purpose.
# This fork is meant to be a direct next experiment from the current 2D U-Net setup,
# not a return to the old 1D conv baseline.

In [ ]:
# =========================
# Plain 2D U-Net generator (no attention bottleneck)
# =========================

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8):
        super().__init__()
        g = min(groups, out_ch)
        while out_ch % g != 0 and g > 1:
            g -= 1
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, k=3, s=1, p=1, groups=groups),
            ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, groups=groups),
        )

    def forward(self, x):
        return self.net(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2, 2), groups=8):
        super().__init__()
        g = min(groups, out_ch)
        while out_ch % g != 0 and g > 1:
            g -= 1
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            ConvGNAct(out_ch, out_ch, k=3, s=1, p=1, groups=groups),
        )

    def forward(self, x):
        return self.net(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)


class MelRefinerUNet2D(nn.Module):
    '''
    Plain mel-domain residual inpainting model.
    Input:
      mel_interp: (B, F, T)
      mask:       (B, 1, T)
    Output:
      delta:      (B, F, T)

    Notes:
      - final conv is zero-initialized, so training starts near identity
      - model predicts a residual, and we only apply it inside the hole
      - this is intentionally the plain 2D U-Net baseline with stronger losses,
        so the effect of loss design can be isolated before architecture changes
    '''
    def __init__(
        self,
        n_mels: int,
        base: int = 24,
        use_mask: bool = True,
        groups: int = 8,
    ):
        super().__init__()
        self.n_mels = n_mels
        self.use_mask = use_mask

        c_in = 1 + (1 if use_mask else 0)

        self.enc1 = DoubleConv(c_in,       base,     groups=groups)
        self.enc2 = DownBlock(base,        base*2,   stride=(2, 2), groups=groups)
        self.enc3 = DownBlock(base*2,      base*4,   stride=(2, 2), groups=groups)
        self.enc4 = DownBlock(base*4,      base*4,   stride=(1, 2), groups=groups)
        self.bot  = DoubleConv(base*4,     base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)

        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)

        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)

        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)

        b  = self.bot(s4)

        u3 = self.up3(b,  s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)

        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1

        delta = self.out_conv(u0).squeeze(1)
        return delta


G = MelRefinerUNet2D(
    n_mels=N_MELS,
    base=G_BASE,
    use_mask=G_USE_MASK,
    groups=G_GROUPS,
).to(device)

print("G params (M):", sum(p.numel() for p in G.parameters()) / 1e6)

with torch.no_grad():
    mel_demo = torch.randn(2, N_MELS, 64, device=device)
    mask_demo = torch.zeros(2, 1, 64, device=device)
    mask_demo[:, :, 10:20] = 1.0
    out_demo = G(mel_demo, mask_demo)
    print("G out shape:", tuple(out_demo.shape))


## Discriminator D - full-band + high-frequency specialist 
This variant keeps the usual mel PatchGAN / multi-scale discriminator, but adds a second discriminator that only sees the upper mel bands.

That means the GAN is no longer only asking:

- “does this whole mel patch look realistic?”

It is also asking:

- “does the repaired upper-band region look realistic?”

which is much closer to the sharp `s / t / transient` failure mode the run care about.


In [ ]:
def set_requires_grad(model, flag: bool):
    if model is None:
        return
    for p in model.parameters():
        p.requires_grad_(flag)

def pack_mel_for_D(mel: torch.Tensor, mask: torch.Tensor):
    """
    mel: (B, n_mels, T)
    mask:(B, 1, T)
    => x: (B, 2, n_mels, T) where channel0=mel, channel1=mask broadcast across mel bins
    """
    x_mel  = mel.unsqueeze(1)  # (B,1,n_mels,T)
    x_mask = mask.unsqueeze(2).expand(-1, 1, mel.shape[1], -1)  # (B,1,n_mels,T)
    return torch.cat([x_mel, x_mask], dim=1)  # (B,2,n_mels,T)

def crop_highband_packed(x: torch.Tensor, start_frac: float = HF_D_START_FRAC):
    """
    x: (B, C, F, T)
    returns only upper mel bins for the HF discriminator
    """
    Fbins = x.shape[-2]
    f0 = int(round(Fbins * start_frac))
    f0 = max(0, min(Fbins - 8, f0))
    return x[:, :, f0:, :]

class MelPatchDiscriminator(nn.Module):
    def __init__(self, in_channels: int = 2, base: int = 32):
        super().__init__()
        SN = nn.utils.spectral_norm

        def conv(in_ch, out_ch, k, s, p):
            return SN(nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p))

        self.blocks = nn.ModuleList([
            nn.Sequential(conv(in_channels, base,   (5, 5), (1, 2), (2, 2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base,        base*2, (5, 5), (2, 2), (2, 2)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base*2,      base*4, (3, 3), (2, 2), (1, 1)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(conv(base*4,      base*8, (3, 3), (2, 2), (1, 1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.out = conv(base*8, 1, (3, 3), (1, 1), (1, 1))

    def forward(self, x):
        fmaps = []
        h = x
        for blk in self.blocks:
            h = blk(h)
            fmaps.append(h)
        return self.out(h), fmaps

class MultiScaleMelDiscriminator(nn.Module):
    def __init__(self, in_channels=2, base=32, scales=(1, 2, 4)):
        super().__init__()
        self.scales = scales
        self.ds = nn.ModuleList([MelPatchDiscriminator(in_channels=in_channels, base=base) for _ in scales])

    def _downsample_time(self, x, s: int):
        if s == 1:
            return x
        return F.avg_pool2d(x, kernel_size=(1, s), stride=(1, s), ceil_mode=False)

    def forward(self, x):
        logits_list = []
        fmaps_list = []
        for D_i, s in zip(self.ds, self.scales):
            xs = self._downsample_time(x, s)
            logits, fmaps = D_i(xs)
            logits_list.append(logits)
            fmaps_list.append(fmaps)
        return logits_list, fmaps_list

def d_loss_lsgan(d_real, d_fake):
    return ((d_real - 1.0) ** 2).mean() + ((d_fake - 0.0) ** 2).mean()

def g_adv_loss_lsgan(d_fake):
    return ((d_fake - 1.0) ** 2).mean()

def fm_loss(fmaps_real, fmaps_fake):
    return sum((fr - ff).abs().mean() for fr, ff in zip(fmaps_real, fmaps_fake))

def d_loss_lsgan_ms(d_reals, d_fakes):
    loss = 0.0
    for dr, df in zip(d_reals, d_fakes):
        loss = loss + ((dr - 1.0) ** 2).mean() + ((df - 0.0) ** 2).mean()
    return loss / len(d_reals)

def g_adv_loss_lsgan_ms(d_fakes):
    loss = 0.0
    for df in d_fakes:
        loss = loss + ((df - 1.0) ** 2).mean()
    return loss / len(d_fakes)

def fm_loss_ms(fmaps_reals, fmaps_fakes):
    loss = 0.0
    n = 0
    for fr_scale, ff_scale in zip(fmaps_reals, fmaps_fakes):
        for fr, ff in zip(fr_scale, ff_scale):
            loss = loss + (fr - ff).abs().mean()
            n += 1
    return loss / max(1, n)

D = None
D_hf = None

if MODE == "A2":
    D = MelPatchDiscriminator(in_channels=2, base=32).to(device)
    if USE_HF_DISCRIMINATOR:
        D_hf = MelPatchDiscriminator(in_channels=2, base=HF_D_BASE).to(device)

if MODE == "A2MS":
    D = MultiScaleMelDiscriminator(in_channels=2, base=32, scales=(1, 2, 4)).to(device)
    if USE_HF_DISCRIMINATOR:
        D_hf = MultiScaleMelDiscriminator(in_channels=2, base=HF_D_BASE, scales=HF_D_SCALES).to(device)

if D is not None:
    print("D params (M):", sum(p.numel() for p in D.parameters())/1e6)
if D_hf is not None:
    print("D_hf params (M):", sum(p.numel() for p in D_hf.parameters())/1e6)


In [ ]:

def _stft_mag(x: torch.Tensor, n_fft: int, hop: int, win: int, eps: float = 1e-7):
    """
    x: (B, T)
    returns magnitude spectrogram: (B, F, TT)
    """
    window = torch.hann_window(win, device=x.device)
    X = torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop,
        win_length=win,
        window=window,
        center=True,
        return_complex=True,
    )
    mag = X.abs().clamp_min(eps)
    return mag

def mrstft_loss(
    x_hat: torch.Tensor,
    x_tgt: torch.Tensor,
    fft_sizes=(512, 1024, 2048),
    hop_sizes=(128, 256, 512),
    win_sizes=(512, 1024, 2048),
    use_logmag=True,
):
    """
    x_hat, x_tgt: (B, T)
    returns scalar tensor
    """
    assert len(fft_sizes) == len(hop_sizes) == len(win_sizes)

    loss = 0.0
    n = 0

    for n_fft, hop, win in zip(fft_sizes, hop_sizes, win_sizes):
        mag_hat = _stft_mag(x_hat, n_fft=n_fft, hop=hop, win=win)
        mag_tgt = _stft_mag(x_tgt, n_fft=n_fft, hop=hop, win=win)

        l_mag = (mag_hat - mag_tgt).abs().mean()

        if use_logmag:
            l_log = (mag_hat.log() - mag_tgt.log()).abs().mean()
            loss = loss + l_mag + l_log
            n += 2
        else:
            loss = loss + l_mag
            n += 1

    return loss / max(1, n)

def waveform_preemphasis(x: torch.Tensor, coeff: float = 0.97):
    """
    x: (B, T)
    """
    if x.shape[-1] < 2:
        return x
    y = x[:, 1:] - coeff * x[:, :-1]
    return F.pad(y, (1, 0))

def waveform_to_mel(wav_bt: torch.Tensor, center: bool = False):
    return mel_spectrogram(
        wav_bt,
        n_fft=h.n_fft,
        num_mels=h.num_mels,
        sampling_rate=h.sampling_rate,
        hop_size=h.hop_size,
        win_size=h.win_size,
        fmin=h.fmin,
        fmax=h.fmax,
        center=center,
    )

def frozen_vocoder_aux_losses(mel_hat: torch.Tensor, mel_tgt: torch.Tensor):
    """
    mel_hat, mel_tgt: (B, n_mels, T)
    returns dict of scalar losses and the generated waveform
    """
    wav_hat = mel_to_wave_vocoder(mel_hat)

    if VOCODER_TARGET_NO_GRAD:
        with torch.no_grad():
            wav_tgt = mel_to_wave_vocoder(mel_tgt)
    else:
        wav_tgt = mel_to_wave_vocoder(mel_tgt)

    loss_voc_mrstft = mrstft_loss(
        wav_hat, wav_tgt,
        fft_sizes=VOC_MRSTFT_FFTS,
        hop_sizes=VOC_MRSTFT_HOPS,
        win_sizes=VOC_MRSTFT_WINS,
        use_logmag=VOC_MRSTFT_LOGMAG,
    )

    mel_hat_rt = waveform_to_mel(wav_hat, center=VOCODER_LOGMEL_CENTER)
    with torch.no_grad():
        mel_tgt_rt = waveform_to_mel(wav_tgt, center=VOCODER_LOGMEL_CENTER)
    loss_voc_logmel = (mel_hat_rt - mel_tgt_rt).abs().mean()

    hp_hat = waveform_preemphasis(wav_hat, coeff=VOCODER_HP_COEFF)
    with torch.no_grad():
        hp_tgt = waveform_preemphasis(wav_tgt, coeff=VOCODER_HP_COEFF)
    loss_voc_hp = (hp_hat - hp_tgt).abs().mean()

    return {
        "wav_hat": wav_hat,
        "wav_tgt": wav_tgt,
        "loss_voc_mrstft": loss_voc_mrstft,
        "loss_voc_logmel": loss_voc_logmel,
        "loss_voc_hp": loss_voc_hp,
    }


In [ ]:
def mel_to_wave_vocoder(mel_bt: torch.Tensor):
    """
    mel_bt: (B, n_mels, T)
    returns waveform: (B, T_wav)
    """
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bt)
    return y.squeeze(1) if y.ndim == 3 and y.shape[1] == 1 else y


## Optimizers + checkpoint save/resume 
This variant starts as a fresh run by default. Resuming from a previous Plain2D checkpoint is handled through the resume configuration.

New in this variant:
- the usual full-band discriminator
- an additional high-frequency discriminator
- separate optimizer state for the HF discriminator


In [ ]:
opt_g = torch.optim.AdamW(G.parameters(), lr=LR_G, betas=(0.9, 0.99), weight_decay=1e-4)

opt_d = None
opt_d_hf = None
if MODE in ["A2", "A2MS"] and D is not None:
    opt_d = torch.optim.AdamW(D.parameters(), lr=LR_D, betas=(0.8, 0.99), weight_decay=1e-4)
if MODE in ["A2", "A2MS"] and D_hf is not None:
    opt_d_hf = torch.optim.AdamW(D_hf.parameters(), lr=LR_D, betas=(0.8, 0.99), weight_decay=1e-4)

def save_ckpt(step: int):
    obj = {
        "step": step,
        "run_config": RUN_CONFIG,
        "mode": MODE,
        "seed": SEED,
        "k_choices": list(K_CHOICES),
        "p_speech": P_SPEECH,
        "G": G.state_dict(),
        "opt_g": opt_g.state_dict(),
        "bigvgan_frontend_config": dict(
            sampling_rate=h.sampling_rate,
            num_mels=h.num_mels,
            n_fft=h.n_fft,
            hop_size=h.hop_size,
            win_size=h.win_size,
            fmin=h.fmin,
            fmax=h.fmax,
        ),
        "bigvgan_model_id": BIGVGAN_MODEL_ID,
        "use_vocoder_aux": USE_VOCODER_AUX,
        "hf_freq_weights": HF_FREQ_WEIGHTS.detach().cpu(),
    }
    if D is not None:
        obj.update({"D": D.state_dict(), "opt_d": opt_d.state_dict()})
    if D_hf is not None:
        obj.update({"D_hf": D_hf.state_dict(), "opt_d_hf": opt_d_hf.state_dict()})

    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)

# Leave as None for a clean fresh run.
RESUME_PATH = None

# Optional same-family auto-resume
if RESUME_PATH is None and AUTO_RESUME_SAME_FAMILY:
    candidates = sorted((DRIVE_ROOT / "checkpoints_runs").glob(f"{RUN_NAME}_*/latest.pt"))
    if candidates:
        RESUME_PATH = candidates[-1]

print("RESUME_PATH:", RESUME_PATH)

start_step = 0
if RESUME_PATH is not None and Path(RESUME_PATH).exists():
    ck = torch.load(str(RESUME_PATH), map_location="cpu")
    G.load_state_dict(ck["G"], strict=True)
    opt_g.load_state_dict(ck["opt_g"])

    if D is not None and ("D" in ck) and ("opt_d" in ck):
        D.load_state_dict(ck["D"], strict=True)
        opt_d.load_state_dict(ck["opt_d"])

    if D_hf is not None and ("D_hf" in ck) and ("opt_d_hf" in ck):
        D_hf.load_state_dict(ck["D_hf"], strict=True)
        opt_d_hf.load_state_dict(ck["opt_d_hf"])

    start_step = int(ck.get("step", 0))

    for pg in opt_g.param_groups:
        pg["lr"] = LR_G * RESUME_LR_SCALE
    if opt_d is not None:
        for pg in opt_d.param_groups:
            pg["lr"] = LR_D * RESUME_LR_SCALE
    if opt_d_hf is not None:
        for pg in opt_d_hf.param_groups:
            pg["lr"] = LR_D * RESUME_LR_SCALE

    print("Resumed:", RESUME_PATH)
    print("start_step:", start_step)
    print("Applied resume LR scale:", RESUME_LR_SCALE)
else:
    print("Starting fresh.")

print("Saving into CKPT_DIR:", CKPT_DIR)

if not LOG_CSV.exists():
    LOG_CSV.write_text(
        "step,lossD,lossD_hf,lossG,loss_recon,loss_hf,loss_tdiff,"
        "loss_adv_full,loss_adv_hf,loss_adv_total,loss_adv_eff,"
        "loss_fm_full,loss_fm_hf,loss_fm_total,adv_w,"
        "base_recon,ref_recon,base_hf,ref_hf,base_tdiff,ref_tdiff,"
        "delta_recon,ratio_recon,win_recon,k,offset,"
        "dr_mean,df_mean,dr_sig,df_sig,"
        "dr_hf_mean,df_hf_mean,dr_hf_sig,df_hf_sig,minutes\n",
        encoding="utf-8"
    )

print("MODE:", MODE)
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("LOG_CSV:", LOG_CSV)


## Training loop 
Main change versus the previous  variant:
- keep the plain 2D U-Net
- keep moderate mel-domain GAN + feature matching
- add frozen BigVGAN auxiliary losses on the waveform generated from `mel_fixed`

So the generator is trained not just to make the mel look better, but also to make the BigVGAN output closer to the target BigVGAN output.


In [ ]:
# =========================
# Training loop (A1 / A2 / A2MS)
# =========================

G.train()
if MODE in ["A2", "A2MS"] and D is not None:
    D.train()
if MODE in ["A2", "A2MS"] and D_hf is not None:
    D_hf.train()

t0 = time.time()

for step in range(start_step + 1, start_step + STEPS + 1):
    wav = next_mixed_batch()

    mel_real = mel_spectrogram(
        wav,
        n_fft=h.n_fft,
        num_mels=h.num_mels,
        sampling_rate=h.sampling_rate,
        hop_size=h.hop_size,
        win_size=h.win_size,
        fmin=h.fmin,
        fmax=h.fmax,
        center=False,
    )

    pair = make_inpainting_pair(mel_real, k_choices=K_CHOICES, randomize_offset=True)
    mel_interp, mask = pair["interp"], pair["mask"]

    with torch.no_grad():
        base_recon = masked_weighted_l1(mel_interp, mel_real, mask, freq_weights=None)
        base_hf    = masked_weighted_l1(mel_interp, mel_real, mask, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=HF_MASK_DILATE)
        base_tdiff = masked_weighted_diff_l1(mel_interp, mel_real, mask, dim=-1, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

    if MODE == "A1":
        delta = G(mel_interp, mask)
        mel_fixed = mel_interp + mask * delta

        missing = mask.sum()
        loss_id = torch.tensor(0.0, device=device)

        if USE_IDENTITY_LOSS_FOR_K0 and (missing < 1.0):
            loss_recon = torch.tensor(0.0, device=device)
            loss_hf    = torch.tensor(0.0, device=device)
            loss_tdiff = torch.tensor(0.0, device=device)
            loss_id    = delta.abs().mean()
        else:
            loss_recon = masked_weighted_l1(mel_fixed, mel_real, mask, freq_weights=None)
            loss_hf    = masked_weighted_l1(mel_fixed, mel_real, mask, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=HF_MASK_DILATE)
            loss_tdiff = masked_weighted_diff_l1(mel_fixed, mel_real, mask, dim=-1, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

        loss_adv_full = torch.tensor(0.0, device=device)
        loss_adv_hf   = torch.tensor(0.0, device=device)
        loss_adv      = torch.tensor(0.0, device=device)
        loss_adv_eff  = torch.tensor(0.0, device=device)
        loss_fm_full  = torch.tensor(0.0, device=device)
        loss_fm_hf    = torch.tensor(0.0, device=device)
        loss_fm       = torch.tensor(0.0, device=device)
        loss_d        = torch.tensor(0.0, device=device)
        loss_d_hf     = torch.tensor(0.0, device=device)
        adv_w = 0.0

        loss_delta = (mask * delta).abs().mean()

        loss_g = (
            (LAMBDA_RECON * loss_recon) +
            (LAMBDA_HF * loss_hf) +
            (LAMBDA_TIME_DIFF * loss_tdiff) +
            (LAMBDA_ID * loss_id) +
            (LAMBDA_DELTA * loss_delta)
        )

        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_g.step()

        dr_mean = df_mean = dr_sig = df_sig = 0.0
        dr_hf_mean = df_hf_mean = dr_hf_sig = df_hf_sig = 0.0

    else:
        do_d_update = (step % D_UPDATE_EVERY) == 0

        if do_d_update:
            set_requires_grad(D, True)
            set_requires_grad(D_hf, True)

            with torch.no_grad():
                delta_ng = G(mel_interp, mask)
                mel_fixed_ng = mel_interp + mask * delta_ng

            x_real = pack_mel_for_D(mel_real, mask)
            x_fake = pack_mel_for_D(mel_fixed_ng.detach(), mask)

            d_real, f_real = D(x_real)
            d_fake, f_fake = D(x_fake)

            if MODE == "A2":
                loss_d = d_loss_lsgan(d_real, d_fake)
            else:
                loss_d = d_loss_lsgan_ms(d_real, d_fake)

            if D_hf is not None:
                x_real_hf = crop_highband_packed(x_real)
                x_fake_hf = crop_highband_packed(x_fake)
                d_real_hf, f_real_hf = D_hf(x_real_hf)
                d_fake_hf, f_fake_hf = D_hf(x_fake_hf)

                if MODE == "A2":
                    loss_d_hf = d_loss_lsgan(d_real_hf, d_fake_hf)
                else:
                    loss_d_hf = d_loss_lsgan_ms(d_real_hf, d_fake_hf)
            else:
                loss_d_hf = torch.tensor(0.0, device=device)

            loss_d_total = loss_d + (HF_D_DLOSS_MULT * loss_d_hf)

            if opt_d is not None:
                opt_d.zero_grad(set_to_none=True)
            if opt_d_hf is not None:
                opt_d_hf.zero_grad(set_to_none=True)
            loss_d_total.backward()
            if D is not None:
                torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
            if D_hf is not None:
                torch.nn.utils.clip_grad_norm_(D_hf.parameters(), 1.0)
            if opt_d is not None:
                opt_d.step()
            if opt_d_hf is not None:
                opt_d_hf.step()

        else:
            loss_d = torch.tensor(0.0, device=device)
            loss_d_hf = torch.tensor(0.0, device=device)
            with torch.no_grad():
                delta_ng = G(mel_interp, mask)
                mel_fixed_ng = mel_interp + mask * delta_ng
                x_real = pack_mel_for_D(mel_real, mask)
                x_fake = pack_mel_for_D(mel_fixed_ng, mask)
                d_real, f_real = D(x_real)
                d_fake, f_fake = D(x_fake)
                if D_hf is not None:
                    x_real_hf = crop_highband_packed(x_real)
                    x_fake_hf = crop_highband_packed(x_fake)
                    d_real_hf, f_real_hf = D_hf(x_real_hf)
                    d_fake_hf, f_fake_hf = D_hf(x_fake_hf)
                else:
                    d_real_hf = d_fake_hf = None

        # G step
        set_requires_grad(D, False)
        set_requires_grad(D_hf, False)

        delta = G(mel_interp, mask)
        mel_fixed = mel_interp + mask * delta

        missing = mask.sum()
        loss_id = torch.tensor(0.0, device=device)

        if USE_IDENTITY_LOSS_FOR_K0 and (missing < 1.0):
            loss_recon = torch.tensor(0.0, device=device)
            loss_hf    = torch.tensor(0.0, device=device)
            loss_tdiff = torch.tensor(0.0, device=device)
            loss_id    = delta.abs().mean()
        else:
            loss_recon = masked_weighted_l1(mel_fixed, mel_real, mask, freq_weights=None)
            loss_hf    = masked_weighted_l1(mel_fixed, mel_real, mask, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=HF_MASK_DILATE)
            loss_tdiff = masked_weighted_diff_l1(mel_fixed, mel_real, mask, dim=-1, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

        d_fake_g, f_fake_g = D(pack_mel_for_D(mel_fixed, mask))
        with torch.no_grad():
            d_real_ng, f_real_ng = D(pack_mel_for_D(mel_real, mask))

        if MODE == "A2":
            loss_adv_full = g_adv_loss_lsgan(d_fake_g)
            loss_fm_full  = fm_loss(f_real_ng, f_fake_g)
        else:
            loss_adv_full = g_adv_loss_lsgan_ms(d_fake_g)
            loss_fm_full  = fm_loss_ms(f_real_ng, f_fake_g)

        if D_hf is not None:
            d_fake_hf_g, f_fake_hf_g = D_hf(crop_highband_packed(pack_mel_for_D(mel_fixed, mask)))
            with torch.no_grad():
                d_real_hf_ng, f_real_hf_ng = D_hf(crop_highband_packed(pack_mel_for_D(mel_real, mask)))

            if MODE == "A2":
                loss_adv_hf = g_adv_loss_lsgan(d_fake_hf_g)
                loss_fm_hf  = fm_loss(f_real_hf_ng, f_fake_hf_g)
            else:
                loss_adv_hf = g_adv_loss_lsgan_ms(d_fake_hf_g)
                loss_fm_hf  = fm_loss_ms(f_real_hf_ng, f_fake_hf_g)
        else:
            loss_adv_hf = torch.tensor(0.0, device=device)
            loss_fm_hf  = torch.tensor(0.0, device=device)

        loss_adv = loss_adv_full + (HF_D_ADV_MULT * loss_adv_hf)
        loss_fm  = loss_fm_full  + (HF_D_FM_MULT  * loss_fm_hf)

        adv_w = LAMBDA_ADV_MAX * min(1.0, step / max(1, ADV_WARMUP_STEPS))
        loss_adv_eff = adv_w * loss_adv
        loss_delta = (mask * delta).abs().mean()

        loss_g = (
            (LAMBDA_RECON * loss_recon) +
            (LAMBDA_HF * loss_hf) +
            (LAMBDA_TIME_DIFF * loss_tdiff) +
            (loss_adv_eff) +
            (LAMBDA_FM * loss_fm) +
            (LAMBDA_ID * loss_id) +
            (LAMBDA_DELTA * loss_delta)
        )

        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_g.step()

        set_requires_grad(D, True)
        set_requires_grad(D_hf, True)

        with torch.no_grad():
            if MODE == "A2":
                dr_logit = d_real
                df_logit = d_fake
            else:
                dr_logit = d_real[0]
                df_logit = d_fake[0]

            dr_mean = float(dr_logit.mean().detach().cpu())
            df_mean = float(df_logit.mean().detach().cpu())
            dr_sig  = float(torch.sigmoid(dr_logit).mean().detach().cpu())
            df_sig  = float(torch.sigmoid(df_logit).mean().detach().cpu())

            if D_hf is not None:
                if MODE == "A2":
                    dr_hf_logit = d_real_hf
                    df_hf_logit = d_fake_hf
                else:
                    dr_hf_logit = d_real_hf[0]
                    df_hf_logit = d_fake_hf[0]
                dr_hf_mean = float(dr_hf_logit.mean().detach().cpu())
                df_hf_mean = float(df_hf_logit.mean().detach().cpu())
                dr_hf_sig  = float(torch.sigmoid(dr_hf_logit).mean().detach().cpu())
                df_hf_sig  = float(torch.sigmoid(df_hf_logit).mean().detach().cpu())
            else:
                dr_hf_mean = df_hf_mean = dr_hf_sig = df_hf_sig = 0.0

    if (step - start_step) % LOG_EVERY == 0:
        dt_min = (time.time() - t0) / 60.0

        with torch.no_grad():
            ref_recon = float(loss_recon.detach().cpu())
            ref_hf    = float(loss_hf.detach().cpu())
            ref_tdiff = float(loss_tdiff.detach().cpu())

            base_r = float(base_recon.detach().cpu())
            base_h = float(base_hf.detach().cpu())
            base_t = float(base_tdiff.detach().cpu())

            delta_recon = base_r - ref_recon
            ratio_recon = improvement_ratio(base_r, ref_recon)
            win_recon   = 1.0 if ref_recon < base_r else 0.0

        print(
            f"step {step:7d} | lossD {float(loss_d):.3f} | lossD_hf {float(loss_d_hf):.3f} | lossG {float(loss_g):.3f} | "
            f"recon {ref_recon:.4f} hf {ref_hf:.4f} tdiff {ref_tdiff:.4f} | "
            f"advFull {float(loss_adv_full):.3f} advHF {float(loss_adv_hf):.3f} effAdv {float(loss_adv_eff):.3f} | "
            f"fmFull {float(loss_fm_full):.3f} fmHF {float(loss_fm_hf):.3f} advW {adv_w:.3f} | "
            f"baseR {base_r:.4f} -> refR {ref_recon:.4f} | baseHF {base_h:.4f} -> refHF {ref_hf:.4f} | "
            f"k={pair['k']} off={pair['offset']} | "
            f"Dmean real/fake {dr_mean:.3f}/{df_mean:.3f} | "
            f"HF-D mean real/fake {dr_hf_mean:.3f}/{df_hf_mean:.3f} | "
            f"{dt_min:.1f} min"
        )

        with LOG_CSV.open("a", encoding="utf-8") as f:
            f.write(
                f"{step},{float(loss_d):.6f},{float(loss_d_hf):.6f},{float(loss_g):.6f},"
                f"{float(loss_recon):.6f},{float(loss_hf):.6f},{float(loss_tdiff):.6f},"
                f"{float(loss_adv_full):.6f},{float(loss_adv_hf):.6f},{float(loss_adv):.6f},{float(loss_adv_eff):.6f},"
                f"{float(loss_fm_full):.6f},{float(loss_fm_hf):.6f},{float(loss_fm):.6f},{adv_w:.6f},"
                f"{base_r:.6f},{ref_recon:.6f},{base_h:.6f},{ref_hf:.6f},{base_t:.6f},{ref_tdiff:.6f},"
                f"{delta_recon:.6f},{ratio_recon:.6f},{win_recon:.0f},"
                f"{pair['k']},{pair['offset']},"
                f"{dr_mean:.6f},{df_mean:.6f},{dr_sig:.6f},{df_sig:.6f},"
                f"{dr_hf_mean:.6f},{df_hf_mean:.6f},{dr_hf_sig:.6f},{df_hf_sig:.6f},{dt_min:.6f}\n"
            )

    if (step - start_step) % CKPT_EVERY == 0:
        save_ckpt(step)

print("done")


## Quick eval on 1 speech + 1 music sample + optional quick listen 
This is still only a sanity check, but it is useful for confirming that:
- the new generator beats interpolation on masked HF losses
- the run did not destabilize
- optional listening still goes through BigVGAN


In [ ]:

@torch.no_grad()
def eval_one_file(path: Path, k=1, offset=0, do_listen=False):
    w = safe_load_wav_mono(path, SR)[: int(SR * SEG_S)].to(device)
    wb = w.unsqueeze(0)

    mel_real = mel_spectrogram(
        wb,
        n_fft=h.n_fft, num_mels=h.num_mels, sampling_rate=h.sampling_rate,
        hop_size=h.hop_size, win_size=h.win_size, fmin=h.fmin, fmax=h.fmax,
        center=False,
    )
    mel_interp, mask = linear_time_inpaint(mel_real, k=k, offset=offset)
    delta = G(mel_interp, mask)
    mel_fixed = mel_interp + mask * delta

    base_recon = masked_weighted_l1(mel_interp, mel_real, mask, freq_weights=None).item()
    ref_recon  = masked_weighted_l1(mel_fixed,  mel_real, mask, freq_weights=None).item()

    base_hf = masked_weighted_l1(mel_interp, mel_real, mask, freq_weights=HF_FREQ_WEIGHTS).item()
    ref_hf  = masked_weighted_l1(mel_fixed,  mel_real, mask, freq_weights=HF_FREQ_WEIGHTS).item()

    base_tdiff = masked_weighted_diff_l1(mel_interp, mel_real, mask, dim=-1, freq_weights=HF_FREQ_WEIGHTS).item()
    ref_tdiff  = masked_weighted_diff_l1(mel_fixed,  mel_real, mask, dim=-1, freq_weights=HF_FREQ_WEIGHTS).item()

    if USE_VOCODER_AUX:
        voc = frozen_vocoder_aux_losses(mel_fixed, mel_real)
        voc_mrstft = float(voc["loss_voc_mrstft"].item())
        voc_logmel = float(voc["loss_voc_logmel"].item())
        voc_hp     = float(voc["loss_voc_hp"].item())
    else:
        voc_mrstft = 0.0
        voc_logmel = 0.0
        voc_hp     = 0.0

    print(
        f"{path.name} | k={k} off={offset} | "
        f"recon {base_recon:.6f}->{ref_recon:.6f} | "
        f"hf {base_hf:.6f}->{ref_hf:.6f} | "
        f"tdiff {base_tdiff:.6f}->{ref_tdiff:.6f} | "
        f"vocSTFT {voc_mrstft:.4f} vocMel {voc_logmel:.4f} vocHP {voc_hp:.4f}"
    )

    if do_listen:
        wav_interp = mel_to_wave_vocoder(mel_interp)[0].detach().cpu().numpy()
        wav_fixed  = mel_to_wave_vocoder(mel_fixed)[0].detach().cpu().numpy()
        wav_real   = mel_to_wave_vocoder(mel_real)[0].detach().cpu().numpy()

        print("Real audio:")
        display(Audio(wav_real, rate=SR))
        print("Interpolated audio:")
        display(Audio(wav_interp, rate=SR))
        print("Fixed audio:")
        display(Audio(wav_fixed, rate=SR))

    return {
        "base_recon": base_recon,
        "ref_recon": ref_recon,
        "base_hf": base_hf,
        "ref_hf": ref_hf,
        "base_tdiff": base_tdiff,
        "ref_tdiff": ref_tdiff,
        "voc_mrstft": voc_mrstft,
        "voc_logmel": voc_logmel,
        "voc_hp": voc_hp,
    }

# Example quick sanity eval
speech_example = read_list(speech_val)[0]
music_example  = read_list(music_val)[0]

print("Speech example:")
_ = eval_one_file(speech_example, k=1, offset=0, do_listen=False)
print()
print("Music example:")
_ = eval_one_file(music_example, k=1, offset=0, do_listen=False)
